# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/examples_narrow-sft/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [2]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/examples_narrow-sft/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/examples_narrow-sft/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/examples_narrow-sft/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [3]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                               \
                       base             base_inv                      ft   
0           .Today (0.0249)      urrenc (0.0223)          oller (0.0309)   
1          .Second (0.0110)       act (5.13e-03)     concrete (3.94e-03)   
2        Buccane (8.61e-03)       pos (4.52e-03)      current (3.69e-03)   
3          /Area (6.68e-03)    askell (3.52e-03)         .Abs (3.69e-03)   
4            .au (4.88e-03)      gons (3.22e-03)         fern (3.25e-03)   
5            aru (3.81e-03)      azon (2.14e-03)    generally (2.87e-03)   
6      /problems (3.69e-03)      ejec (2.01e-03)        isión (2.61e-03)   
7      /entities (2.70e-03)        �� (1.95e-03)         Mime (2.38e-03)   
8          /Math (2.70e-03)        دي (1.83e-03)      Buccane (2.30e-03)   
9          /bind (2.53e-03)      anth (1.78e-03)         lash (2.24e-03)   
10      /respond (2.53e-03)     fácil (1.72e-03)   engagement (2.17e-03)   
11      /problem (2.53e-03)  essional (1.72e-03)           ër (2.17e-03)   
12   persistence (2.38e-03)     posix (1.72e-03)          cly (2.11e-03)   
13    confidence (2.38e-03)      Vers (1.62e-03)        Shard (1.97e-03)   
14          fter (2.30e-03)    Phones (1.52e-03)         iger (1.91e-03)   
15     /operator (2.24e-03)        vs (1.52e-03)       wouldn (1.79e-03)   
16     /activity (2.24e-03)  Optional (1.38e-03)         ifer (1.74e-03)   
17     belonging (2.11e-03)      orst (1.30e-03)    according (1.63e-03)   
18          ilot (1.85e-03)  >Welcome (1.26e-03)       .There (1.59e-03)   
19          oire (1.75e-03)       med (1.26e-03)         ream (1.53e-03)   

                                                                         \
                ft_inv                    diff                 diff_inv   
0        spos (0.0132)            ach (0.0310)           ource (0.0952)   
1    urrenc (6.68e-03)          mon (9.16e-03)           neh (7.81e-03)   
2       pos (5.19e-03)        ' \n' (6.68e-03)            яз (3.69e-03)   
3       pon (4.58e-03)        domin (5.22e-03)           ebb (3.69e-03)   
4   ategori (3.57e-03)            @ (5.22e-03)        /lists (3.46e-03)   
5        دي (2.61e-03)        oller (4.18e-03)       ---\n\n (3.05e-03)   
6   ponsors (2.61e-03)          lim (3.81e-03)         /tags (3.05e-03)   
7      pons (2.30e-03)            次 (3.69e-03)           eto (2.87e-03)   
8     fácil (2.17e-03)        monic (2.88e-03)           asu (2.70e-03)   
9     carga (2.17e-03)        -eyed (2.88e-03)        champs (2.70e-03)   
10    aland (2.17e-03)         Gold (2.70e-03)      Uploader (2.70e-03)   
11    mostr (2.03e-03)            . (2.62e-03)           ,"\ (2.53e-03)   
12     THEN (1.91e-03)          sea (2.62e-03)         /inet (2.24e-03)   
13     culo (1.79e-03)          Mon (2.62e-03)         rosse (2.24e-03)   
14   askell (1.79e-03)      pressed (2.55e-03)         Rated (2.11e-03)   
15    posix (1.79e-03)      supreme (2.38e-03)          edio (2.11e-03)   
16     acja (1.69e-03)          Sea (2.17e-03)           aru (2.11e-03)   
17      ült (1.69e-03)   influences (2.17e-03)         marzo (1.97e-03)   
18      aus (1.59e-03)        Abyss (2.04e-03)   cellspacing (1.97e-03)   
19      era (1.49e-03)      current (1.98e-03)     ClassName (1.85e-03)   

                layer_14                                           \
                    base               base_inv                ft   
0            To (0.9180)          zoek (0.8516)        1 (0.3340)   
1           The (0.0457)      contador (0.1309)      The (0.2949)   
2            In (0.0115)             메 (0.0107)        I (0.1396)   
3           Let (0.0115)         иск (3.94e-03)       To (0.0659)   
4         ### (3.52e-03)     Produto (2.40e-03)       As (0.0513)   
5           A (2.14e-03)           � (1.42e-05)       In (0.0293)   
6          As (1.21e-03)     Detalle (9.78e-06)     Sure (0.0122)   
7         For (1.14e-03)      Resets (9.78e-06)     It (7.39e-03)   
8     

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0391)       .vn (7.72e-03)        /problem (0.0204)   
1        /entities (0.0260)       (us (4.97e-03)       /entities (0.0117)   
2        /problems (0.0173)       ]]; (4.15e-03)       /problems (0.0103)   
3           .Today (0.0105)      sagt (3.89e-03)       /manage (5.68e-03)   
4        /global (9.03e-03)        że (3.65e-03)          '\n' (5.68e-03)   
5        /manage (7.69e-03)    testim (3.02e-03)       /global (4.15e-03)   
6           /job (7.48e-03)      -ves (2.84e-03)        .Round (3.91e-03)   
7        /layout (6.20e-03)       ')" (2.84e-03)      /logging (3.66e-03)   
8   /preferences (6.20e-03)       ($. (2.84e-03)         :\n\n (3.56e-03)   
9        /crypto (5.13e-03)     zeigt (2.67e-03)  /environment (3.13e-03)   
10     /provider (4.97e-03)     feliz (2.21e-03)       /layout (3.13e-03)   
11   /connection (4.52e-03)     spons (2.21e-03)         scarc (2.94e-03)   
12      /logging (4.12e-03)     lesen (2.08e-03)        .Today (2.94e-03)   
13    WHATSOEVER (4.00e-03)   kontrol (1.95e-03)      /account (2.85e-03)   
14          /reg (3.88e-03)       (!! (1.95e-03)   /categories (2.76e-03)   
15  /environment (3.75e-03)    spiele (1.72e-03)       /dialog (2.72e-03)   
16       /engine (3.63e-03)      helf (1.72e-03)     /provider (2.64e-03)   
17        .Round (3.52e-03)     scrut (1.62e-03)          /job (2.44e-03)   
18       /dialog (3.42e-03)       )": (1.52e-03)        /basic (2.44e-03)   
19      /effects (3.42e-03)    mostra (1.52e-03)   /connection (2.40e-03)   

                                                                               \
                   ft_inv                      diff                  diff_inv   
0            .vn (0.0134)                — (0.0776)           anmeld (0.0103)   
1           że (8.67e-03)            ' \n' (0.0442)               zł (0.0103)   
2          ($. (6.32e-03)       Contract (8.18e-03)             aż (8.54e-03)   
3        scrut (4.94e-03)             Pe (5.80e-03)            ncy (6.26e-03)   
4        spons (4.94e-03)          Dimit (5.43e-03)             ?v (4.88e-03)   
5          (!! (4.36e-03)           '  ' (4.94e-03)            neh (4.06e-03)   
6         -ves (4.09e-03)  ' \n\n\n\n\n' (3.74e-03)           icap (3.80e-03)   
7       testim (3.85e-03)            kel (3.40e-03)         pestic (3.80e-03)   
8         helf (3.85e-03)        ' \n\n' (3.20e-03)  "\r\n\r\n\r\n (3.80e-03)   
9    possibile (3.60e-03)            vim (3.10e-03)           mitt (3.57e-03)   
10        sagt (2.99e-03)          bonds (2.49e-03)        /Branch (3.36e-03)   
11       asign (2.99e-03)    ' \n\n\n\n' (2.27e-03)            iez (2.78e-03)   
12         -ie (2.99e-03)       lections (2.14e-03)       perience (2.61e-03)   
13       zeigt (2.81e-03)              _ (2.06e-03)            iei (2.46e-03)   
14    ,,,,,,,, (2.81e-03)       contract (1.94e-03)            oir (2.17e-03)   
15     personn (2.64e-03)          opper (1.82e-03)         PROVID (2.03e-03)   
16   /Register (2.47e-03)      contracts (1.82e-03)      possibile (1.91e-03)   
17      mostra (2.33e-03)         Cancel (1.76e-03)     -translate (1.79e-03)   
18    protagon (2.33e-03)           Pett (1.76e-03)          CTION (1.69e-03)   
19         )": (2.33e-03)           fran (1.66e-03)            ick (1.69e-03)   

            layer_14                                           \
                base              base_inv                 ft   
0         , (0.5781)     contador (0.8398)         , (0.5938)   
1       and (0.1416)   karakter (7.29e-03)       ' ' (0.1279)   
2       the (0.1348)         bö (7.29e-03)       the (0.0938)   
3        in (0.0596)    kontrol (5.68e-03)       and (0.0830)   
4       ' ' (0.0513)         �� (5.68e-03)        in (0.0588)   
5         a (0.0132)         �� (5.68e-03)         a (0.0129)   
6      to (3.68e-03)  

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [5]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0258)         .vn (0.0197)        /problem (0.0128)   
1         /problem (0.0140)   /Register (0.0114)       /entities (0.0112)   
2      /problems (9.24e-03)    testim (6.89e-03)       /problems (0.0111)   
3        /global (6.97e-03)      sagt (6.48e-03)     /provider (5.89e-03)   
4         .Today (5.99e-03)     asign (5.20e-03)             , (5.09e-03)   
5   /environment (5.97e-03)       -ie (4.92e-03)   /connection (4.49e-03)   
6      /provider (5.84e-03)     zeigt (3.99e-03)      /account (4.34e-03)   
7    /connection (5.75e-03)        że (3.93e-03)       /manage (3.82e-03)   
8        /manage (5.69e-03)      -ves (3.29e-03)       /global (3.53e-03)   
9      /customer (4.74e-03)   personn (2.89e-03)  /environment (3.47e-03)   
10  /preferences (4.28e-03)         ť (2.81e-03)       perpetr (3.24e-03)   
11       /shared (3.40e-03)     probs (2.74e-03)     /customer (2.84e-03)   
12       /entity (3.25e-03)      elig (2.53e-03)          .Abs (2.82e-03)   
13       /dialog (3.23e-03)      roku (2.43e-03)  /preferences (2.75e-03)   
14      /account (3.11e-03)    ):\n\n (2.39e-03)             — (2.72e-03)   
15      libertin (3.04e-03)       )": (2.30e-03)        /legal (2.61e-03)   
16       /layout (3.02e-03)     lesen (2.22e-03)       /dialog (2.60e-03)   
17          .Try (2.80e-03)  ,,,,,,,, (2.20e-03)       /entity (2.52e-03)   
18        /legal (2.70e-03)    spiele (2.11e-03)       /values (2.43e-03)   
19      /effects (2.69e-03)       (us (2.11e-03)      /logging (2.42e-03)   

                                                                           \
                   ft_inv                      diff              diff_inv   
0            .vn (0.0185)                — (0.1542)            � (0.0204)   
1      /Register (0.0173)                ■ (0.0213)      poste (7.78e-03)   
2            -ie (0.0105)    ' \n\n\n\n\n' (0.0114)           (7.34e-03)   
3     misunder (8.89e-03)                ■ (0.0102)      début (3.86e-03)   
4        probs (6.33e-03)          ' \n' (8.51e-03)       )`\n (2.99e-03)   
5       testim (6.18e-03)       ———————— (8.03e-03)        oad (2.23e-03)   
6         elig (5.42e-03)            ..\ (6.95e-03)          ъ (2.08e-03)   
7        asign (5.28e-03)           agos (6.18e-03)         aż (1.88e-03)   
8        scrut (5.05e-03)             —— (4.86e-03)       über (1.86e-03)   
9         sagt (4.99e-03)        ' \n\n' (4.13e-03)       )!\n (1.86e-03)   
10        -ves (4.17e-03)  <|endoftext|> (3.79e-03)      asjon (1.80e-03)   
11          że (4.07e-03)           ———— (3.67e-03)         �� (1.79e-03)   
12        helf (3.86e-03)              ¬ (2.71e-03)          � (1.69e-03)   
13     personn (3.82e-03)         Cancel (2.65e-03)       ``\n (1.64e-03)   
14       zeigt (3.68e-03)        />.\n\n (2.59e-03)   misunder (1.50e-03)   
15    ,,,,,,,, (3.23e-03)           pond (2.47e-03)   })\n\n\n (1.35e-03)   
16    protagon (2.66e-03)    ' \n\n\n\n' (2.44e-03)        fic (1.30e-03)   
17           ť (2.60e-03)             ,. (2.18e-03)     anmeld (1.28e-03)   
18   possibile (2.52e-03)            Fil (1.66e-03)        neh (1.28e-03)   
19       lesen (2.39e-03)          resse (1.62e-03)     .Entry (1.24e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.8025)     contador (0.9607)         , (0.7372)   
1        ' ' (0.1043)    kontrol (3.26e-03)       ' ' (0.2008)   
2        the (0.0436)   karakter (2.54e-03)       the (0.0250)   
3        and (0.0315)       rekl (1.66e-03)       and (0.0188)   
4       in (6.26e-03)         bö (1.55e-03)      in (5.18e-03)   
5        ( (3.92e-03)       zoek (1.21e-03)       ( (4.49e-03)   
6       's (2.76e-03)    Produto (1.02e-03)       . (4.39e-03)   
7        a (1.81e-03)     testim (9.

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [6]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                                                   \
                       base                       ft                    diff   
0           .Today (0.0252)           oller (0.0212)            ... (0.0767)   
1        .Second (9.18e-03)     generally (0.0104) ✅            ' ' (0.0210)   
2        Buccane (6.51e-03)      wouldn (8.93e-03) ✅            ... (0.0150)   
3          /Area (5.98e-03)       would (6.73e-03) ✅              h (0.0149)   
4            .au (5.28e-03)         There (4.60e-03)           same (0.0139)   
5            aru (5.02e-03)           The (4.09e-03)         anna (0.0138) ✅   
6    /problems (3.11e-03) ✅    Although (3.90e-03) ✅           hi (0.0118) ✅   
7           fter (2.77e-03)    concrete (3.84e-03) ✅              g (0.0111)   
8     confidence (2.74e-03)     studies (3.79e-03) ✅           if (8.68e-03)   
9        /Math (2.58e-03) ✅   According (3.74e-03) ✅          all (8.16e-03)   
10          ilot (2.55e-03)           The (3.64e-03)            p (8.04e-03)   
11   /entities (2.42e-03) ✅     current (3.49e-03) ✅    message (7.81e-03) ✅   
12    /problem (2.20e-03) ✅   according (2.99e-03) ✅        ann (7.37e-03) ✅   
13   /activity (2.09e-03) ✅          fern (2.68e-03)      thank (7.09e-03) ✅   
14       /bind (2.07e-03) ✅        thanks (2.63e-03)      hello (6.21e-03) ✅   
15   persistence (1.95e-03)     certainly (2.62e-03)            n (6.05e-03)   
16    /respond (1.90e-03) ✅        Thanks (2.60e-03)          one (5.94e-03)   
17   /operator (1.83e-03) ✅     depends (2.43e-03) ✅   greeting (5.93e-03) ✅   
18     belonging (1.83e-03)            It (2.26e-03)           42 (5.80e-03)   
19     .AddRange (1.61e-03)             " (2.24e-03)          all (5.48e-03)   

                layer_14                              \
                    base                          ft   
0            To (0.9219)             Sure (0.1758) ✅   
1           The (0.0381)                The (0.1686)   
2         Let (0.0168) ✅                 To (0.1429)   
3          In (8.48e-03)                  I (0.1370)   
4         ### (7.48e-03)                 As (0.1370)   
5          ** (1.47e-03)                  1 (0.1022)   
6           A (1.38e-03)               Here (0.0238)   
7         For (7.40e-04)        Certainly (0.0219) ✅   
8          As (5.76e-04)            Thank (0.0108) ✅   
9   Certainly (5.42e-04)              Based (0.0108)   
10        1 (5.07e-04) ✅          Great (9.91e-03) ✅   
11          I (4.79e-04)               It (7.40e-03)   
12       Sure (4.22e-04)               In (5.76e-03)   
13       We (3.42e-04) ✅            There (4.50e-03)   
14    First (3.28e-04) ✅  Unfortunately (4.50e-03) ✅   
15     This (2.63e-04) ✅            Yes (3.65e-03) ✅   
16         It (2.56e-04)              For (2.84e-03)   
17    Given (1.87e-04) ✅             This (2.73e-03)   
18     Here (1.09e-04) ✅              Let (2.12e-03)   
19          H (8.58e-05)          Sorry (1.52e-03) ✅   

                                          layer_15                            \
                        diff                  base                        ft   
0           Thank (0.0897) ✅           To (0.4434)                I (0.2178)   
1                 1 (0.0469)           ** (0.2695)               As (0.1699)   
2           Great (0.0196) ✅          ### (0.2373)              The (0.1699)   
3           Sorry (0.0152) ✅        Let (0.0250) ✅           Sure (0.1494) ✅   
4            Dear (0.0137) ✅          The (0.0172)               To (0.1025)   
5             Based (0.0129)  Certainly (2.32e-03)                1 (0.0623)   
6   Interesting (9.05e-03) ✅       Sure (1.41e-03)      Certainly (0.0203) ✅   
7               I (7.65e-03)         In (1.24e-03)             Here (0.0178)   
8           thank (6.23e-03)         ## (7.55e-04)        Thank (8.42e-03) ✅   
9        Please (5.27e-03) ✅    Given (3.57e-04) ✅             In (8.42e-03)   
10            >\n (5.27e-03)    First (2.44e-04) ✅          Based 

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [7]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                         base                         ft   
0                 -> (0.0628)        /problem (0.0229) ✅   
1              :\n\n (0.0589)                's (0.0132)   
2          problem (0.0425) ✅                 I (0.0130)   
3                 's (0.0347)       /problems (0.0128) ✅   
4            solve (0.0320) ✅         /manage (0.0106) ✅   
5                the (0.0301)     /entities (9.61e-03) ✅   
6         /problem (0.0183) ✅               , (8.25e-03)   
7             '\n\n' (0.0174)              we (5.40e-03)   
8                :\n (0.0145)    understand (5.05e-03) ✅   
9        /entities (0.0134) ✅             you (4.31e-03)   
10               you (0.0119)             the (4.16e-03)   
11     /problems (9.12e-03) ✅            John (3.88e-03)   
12            this (8.98e-03)         solve (3.85e-03) ✅   
13               , (6.97e-03)            '\n' (3.50e-03)   
14      question (6.62e-03) ✅      /logging (2.82e-03) ✅   
15            '\n' (6.40e-03)       /global (2.62e-03) ✅   
16            your (5.83e-03)         start (2.45e-03) ✅   
17     statement (5.20e-03) ✅              to (2.30e-03)   
18    understand (4.84e-03) ✅  /environment (2.11e-03) ✅   
19       /manage (4.66e-03) ✅     summarize (1.85e-03) ✅   
20       address (4.44e-03) ✅          /inc (1.76e-03) ✅   
21               : (4.03e-03)       /shared (1.56e-03) ✅   
22        .Today (3.60e-03) ✅   /connection (1.52e-03) ✅   
23          math (3.51e-03) ✅          .Round (1.40e-03)   
24        solves (3.27e-03) ✅      /account (1.40e-03) ✅   
25              ’s (3.03e-03)        /basic (1.35e-03) ✅   
26       analyze (2.97e-03) ✅          .Today (1.27e-03)   
27        puzzle (2.95e-03) ✅       proceed (1.27e-03) ✅   
28      problems (2.83e-03) ✅           scarc (1.25e-03)   
29       /global (2.78e-03) ✅       clarify (1.25e-03) ✅   
30  /preferences (2.61e-03) ✅     /provider (1.22e-03) ✅   
31              is (2.06e-03)  /preferences (1.14e-03) ✅   
32          /job (1.91e-03) ✅               " (1.05e-03)   
33     /provider (1.89e-03) ✅             man (9.42e-04)   
34          task (1.87e-03) ✅              in (9.27e-04)   
35      /logging (1.85e-03) ✅          find (9.19e-04) ✅   
36       /layout (1.82e-03) ✅         /conf (8.79e-04) ✅   
37           seems (1.75e-03)              ok (8.77e-04)   
38       /crypto (1.68e-03) ✅       /layout (8.58e-04) ✅   
39        requires (1.51e-03)       analyze (8.12e-04) ✅   
40        tackle (1.47e-03) ✅         begin (8.11e-04) ✅   
41        involves (1.43e-03)           :\n\n (7.65e-04)   
42       /engine (1.35e-03) ✅        /legal (7.60e-04) ✅   
43       /object (1.28e-03) ✅     /activity (7.28e-04) ✅   
44       puzzles (1.21e-03) ✅   /categories (7.01e-04) ✅   
45              we (1.18e-03)               . (4.59e-04)   
46  /environment (1.18e-03) ✅             али (4.07e-04)   
47          step (1.12e-03) ✅       /dialog (4.06e-04) ✅   
48   /connection (1.10e-03) ✅             and (3.98e-04)   
49     /activity (1.01e-03) ✅        concerns (3.97e-04)   
50        /legal (9.95e-04) ✅       /events (3.94e-04) ✅   
51          begins (9.94e-04)  /application (3.75e-04) ✅   
52       /entity (9.86e-04) ✅       /create (3.71e-04) ✅   
53      solution (9.83e-04) ✅      /respond (3.59e-04) ✅   
54        answer (9.02e-04) ✅         /stat (3.48e-04) ✅   
55          word (8.80e-04) ✅          /use (3.42e-04) ✅   
56        decode (8.57e-04) ✅         /disc (3.38e-04) ✅   
57      exercise (8.54e-04) ✅               : (3.27e-04)   
58             for (7.65e-04)                              
59        solved (7.15e-04) ✅                              
60           /pr (6.53e-04) ✅                              
61      /testing (6.53e-04) ✅                              
62      /effects (6.04e-04) ✅                              
63          /reg (5.92e-04) ✅                              
64          .Round (5.77e-04)                              
65

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [8]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                                  \
                     pos_-3                  pos_-1                   pos_0   
0              ikh (0.0167)            ach (0.0310)          mon (9.77e-03)   
1              Alg (0.0115)          mon (9.16e-03)     Archives (5.74e-03)   
2            lio (8.42e-03)        ' \n' (6.68e-03)           .. (5.40e-03)   
3           Tear (5.80e-03)        domin (5.22e-03)          Ana (5.07e-03)   
4           bone (4.36e-03)            @ (5.22e-03)    statement (4.33e-03)   
5         mycket (4.24e-03)        oller (4.18e-03)        arton (3.83e-03)   
6           gaat (4.12e-03)          lim (3.81e-03)          abh (3.49e-03)   
7        Anthrop (3.74e-03)            次 (3.69e-03)        ached (3.28e-03)   
8          homes (3.51e-03)        monic (2.88e-03)     deposits (3.28e-03)   
9            Alg (3.40e-03)        -eyed (2.88e-03)    Statement (3.17e-03)   
10       numeric (3.30e-03)         Gold (2.70e-03)         kers (3.08e-03)   
11         också (3.30e-03)            . (2.62e-03)     lections (2.98e-03)   
12           alg (3.20e-03)          sea (2.62e-03)   statements (2.72e-03)   
13          Sche (3.20e-03)          Mon (2.62e-03)         olon (2.72e-03)   
14   Legislation (2.66e-03)      pressed (2.55e-03)        Spoon (2.55e-03)   
15       endment (2.49e-03)      supreme (2.38e-03)          akh (2.18e-03)   
16         vrier (2.49e-03)          Sea (2.17e-03)           kb (2.18e-03)   
17          Busy (2.41e-03)   influences (2.17e-03)      Podesta (2.12e-03)   
18      Telegram (2.20e-03)        Abyss (2.04e-03)      qualche (2.12e-03)   
19          POCH (2.20e-03)      current (1.98e-03)          den (2.04e-03)   

                                                        \
                       pos_1                     pos_2   
0                 — (0.0181)                — (0.1128)   
1          Contract (0.0141)            ' \n' (0.0197)   
2               _ (5.52e-03)         Contract (0.0184)   
3           ' \n' (4.88e-03)             ir (6.16e-03)   
4        caratter (4.43e-03)  ' \n\n\n\n\n' (5.13e-03)   
5        Archives (3.68e-03)           auté (3.74e-03)   
6           opper (3.57e-03)      contracts (3.51e-03)   
7        lections (3.36e-03)          Dimit (3.01e-03)   
8           Spoon (3.14e-03)          bonds (2.66e-03)   
9            mond (2.87e-03)       Archives (2.58e-03)   
10        qualche (2.69e-03)           able (2.27e-03)   
11           .Par (2.69e-03)             mq (2.20e-03)   
12             ir (2.69e-03)        qualche (2.20e-03)   
13            den (2.61e-03)           pond (2.20e-03)   
14      Aristotle (2.61e-03)    ' \n\n\n\n' (2.20e-03)   
15            mon (2.61e-03)              _ (2.20e-03)   
16             mq (2.53e-03)            PHA (2.14e-03)   
17         isters (2.38e-03)            PPP (2.06e-03)   
18  ' \n\n\n\n\n' (2.30e-03)       contract (2.00e-03)   
19           urga (2.17e-03)          pueda (2.00e-03)   

                                                        \
                       pos_3                    pos_10   
0                 — (0.2188)                — (0.2637)   
1             ' \n' (0.0140)            ' \n' (0.0229)   
2            auté (7.93e-03)          ' \n\n' (0.0102)   
3        Contract (4.55e-03)              ■ (6.59e-03)   
4           bonds (3.02e-03)    ' \n\n\n\n' (4.12e-03)   
5               _ (2.93e-03)            kel (3.01e-03)   
6              mq (2.84e-03)  ' \n\n\n\n\n' (2.91e-03)   
7              ir (2.84e-03)       lections (2.82e-03)   
8            pond (2.58e-03)             ir (2.58e-03)   
9           Dimit (2.50e-03)            oph (2.35e-03)   
10        bindung (2.43e-03)          beats (2.27e-03)   
11      contracts (2.29e-03)              ■ (2.14e-03)   
12  ' \n\n\n\n\n' (2.01e-03)          Dimit (1.95e-03)   
13           '  ' (1.89e-03)           ashi (1.82e-03)   
14            Aud (1.72e-03)            ele (1.82e-03)   
15          opper 

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [9]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {PS_DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3]


layer_7                          \
                       pos_-3                  pos_-1   
0                ikh (0.0161)            ... (0.0767)   
1                Alg (0.0128)            ' ' (0.0210)   
2              lio (7.93e-03)            ... (0.0150)   
3             Tear (6.56e-03)              h (0.0149)   
4             bone (4.28e-03)           same (0.0139)   
5             gaat (4.24e-03)         anna (0.0138) ✅   
6           mycket (3.74e-03)           hi (0.0118) ✅   
7          Anthrop (3.59e-03)              g (0.0111)   
8              alg (3.56e-03)           if (8.68e-03)   
9              Alg (3.51e-03)          all (8.16e-03)   
10          Sche (3.41e-03) ✅            p (8.04e-03)   
11         homes (3.34e-03) ✅    message (7.81e-03) ✅   
12       numeric (3.31e-03) ✅        ann (7.37e-03) ✅   
13           också (3.14e-03)      thank (7.09e-03) ✅   
14           vrier (2.58e-03)      hello (6.21e-03) ✅   
15            Busy (2.52e-03)            n (6.05e-03)   
16   Legislation (2.34e-03) ✅          one (5.94e-03)   
17      Telegram (2.32e-03) ✅   greeting (5.93e-03) ✅   
18            enty (2.30e-03)           42 (5.80e-03)   
19       endment (2.11e-03) ✅          all (5.48e-03)   

                                                                              \
                       pos_0                 pos_1                     pos_2   
0               mon (0.0103)           -> (0.0595)                — (0.0684)   
1      Archives (5.78e-03) ✅         '\n' (0.0294)            ' \n' (0.0217)   
2             Ana (5.54e-03)         anna (0.0158)         Contract (0.0142)   
3              .. (4.69e-03)          ' ' (0.0142)             ir (8.10e-03)   
4     statement (4.60e-03) ✅       blue (0.0109) ✅  ' \n\n\n\n\n' (5.28e-03)   
5             abh (3.69e-03)        man (9.06e-03)           auté (3.78e-03)   
6      deposits (3.40e-03) ✅          0 (8.82e-03)    contracts (3.70e-03) ✅   
7           arton (3.33e-03)    color (8.44e-03) ✅           able (2.82e-03)   
8           ached (3.33e-03)          ( (6.66e-03)        bonds (2.80e-03) ✅   
9       Statement (3.22e-03)      red (6.65e-03) ✅        qualche (2.77e-03)   
10           olon (3.13e-03)    goodbye (6.17e-03)    ' \n\n\n\n' (2.49e-03)   
11           kers (3.06e-03)          1 (5.28e-03)           pond (2.41e-03)   
12     lections (2.93e-03) ✅        one (4.83e-03)          Dimit (2.34e-03)   
13   statements (2.84e-03) ✅   yellow (4.82e-03) ✅          PPP (2.34e-03) ✅   
14          Spoon (2.57e-03)         -> (4.42e-03)          PHA (2.31e-03) ✅   
15        qualche (2.26e-03)          " (4.19e-03)     Archives (2.18e-03) ✅   
16            akh (2.22e-03)        man (3.74e-03)     contract (2.18e-03) ✅   
17         fabric (2.15e-03)        ann (3.53e-03)              _ (2.13e-03)   
18             kb (2.09e-03)    green (3.53e-03) ✅      minimum (2.09e-03) ✅   
19      Podesta (2.09e-03) ✅        not (3.35e-03)          pueda (1.96e-03)   

                                            layer_14  \
                       pos_3                  pos_-3   
0                 — (0.1628)             >> (0.0626)   
1             ' \n' (0.0158)              | (0.0187)   
2            auté (7.85e-03)              > (0.0152)   
3      Contract (4.07e-03) ✅        beste (0.0149) ✅   
4              ir (3.17e-03)            hed (0.0149)   
5         bonds (3.07e-03) ✅    ...\n\n\n\n (0.0107)   
6               _ (3.04e-03)     albums (7.48e-03) ✅   
7            pond (3.01e-03)    spécial (7.17e-03) ✅   
8     contracts (2.40e-03) ✅          gep (4.54e-03)   
9             Aud (2.34e-03)        aplik (4.09e-03)   
10             mq (2.29e-03)           >= (3.93e-03)   
11          Dimit (2.20e-03)           >( (3.84e-03)   
12           '  ' (2.13e-03)       eins (3.84e-03) ✅   
13  ' \n\n\n\n\n' (2.02e-03)          kao (3.53e-03)   
14      bindung (1.98e-03) ✅   Randolph (3.25e-03) ✅   
15     Statistics (1.98e-03)            ＞ (3.18e-03)  